In [1]:
import torch

import torch.nn as nn
import torch.nn.functional as F


class BahdanauAttention(nn.Module):
    """
    Additive attention:
    score(q, k) = v^T tanh(W_q q + W_k k)
    """
    def __init__(self, hidden_dim: int, attn_dim: int = None):
        super().__init__()
        attn_dim = attn_dim or hidden_dim
        self.W_q = nn.Linear(hidden_dim, attn_dim, bias=False)
        self.W_k = nn.Linear(hidden_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, query, keys, values=None, mask=None):
        """
        query:  [B, H]
        keys:   [B, T, H]
        values: [B, T, H] (default=keys)
        mask:   [B, T] (True=valid, False=pad)
        """
        if values is None:
            values = keys

        q = self.W_q(query).unsqueeze(1)     # [B,1,A]
        k = self.W_k(keys)                   # [B,T,A]
        energy = self.v(torch.tanh(q + k)).squeeze(-1)  # [B,T]

        if mask is not None:
            energy = energy.masked_fill(~mask, float("-inf"))

        attn_weights = F.softmax(energy, dim=-1)  # [B,T]
        context = torch.bmm(attn_weights.unsqueeze(1), values).squeeze(1)  # [B,H]
        return context, attn_weights

In [2]:
class LuongAttention(nn.Module):
    """
    Multiplicative attention:
    - dot:     score(q,k)=q^T k
    - general: score(q,k)=q^T W k
    - concat:  score(q,k)=v^T tanh(W [q;k])  (Luong paper variant)
    """
    def __init__(self, hidden_dim: int, score_type: str = "dot"):
        super().__init__()
        assert score_type in {"dot", "general", "concat"}
        self.score_type = score_type
        self.hidden_dim = hidden_dim

        if score_type == "general":
            self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        elif score_type == "concat":
            self.W = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
            self.v = nn.Linear(hidden_dim, 1, bias=False)

    def _score(self, query, keys):
        # query: [B,H], keys: [B,T,H]
        if self.score_type == "dot":
            return torch.bmm(keys, query.unsqueeze(-1)).squeeze(-1)  # [B,T]

        if self.score_type == "general":
            k_proj = self.W(keys)  # [B,T,H]
            return torch.bmm(k_proj, query.unsqueeze(-1)).squeeze(-1)  # [B,T]

        # concat
        B, T, H = keys.shape
        q_expand = query.unsqueeze(1).expand(B, T, H)
        x = torch.cat([q_expand, keys], dim=-1)         # [B,T,2H]
        return self.v(torch.tanh(self.W(x))).squeeze(-1)  # [B,T]

    def forward(self, query, keys, values=None, mask=None):
        if values is None:
            values = keys

        energy = self._score(query, keys)  # [B,T]
        if mask is not None:
            energy = energy.masked_fill(~mask, float("-inf"))

        attn_weights = F.softmax(energy, dim=-1)  # [B,T]
        context = torch.bmm(attn_weights.unsqueeze(1), values).squeeze(1)  # [B,H]
        return context, attn_weights

In [3]:
torch.manual_seed(7)

B, T, H = 2, 5, 16
query = torch.randn(B, H)
keys = torch.randn(B, T, H)
values = torch.randn(B, T, H)
mask = torch.tensor([[1, 1, 1, 1, 0], [1, 1, 0, 0, 0]], dtype=torch.bool)

bahdanau = BahdanauAttention(H)
luong_dot = LuongAttention(H, "dot")
luong_general = LuongAttention(H, "general")

c_bah, w_bah = bahdanau(query, keys, values, mask)
c_dot, w_dot = luong_dot(query, keys, values, mask)
c_gen, w_gen = luong_general(query, keys, values, mask)

print("Bahdanau context shape:", c_bah.shape, "weights shape:", w_bah.shape)
print("Luong(dot) context shape:", c_dot.shape, "weights shape:", w_dot.shape)
print("Luong(general) context shape:", c_gen.shape, "weights shape:", w_gen.shape)

print("Bahdanau params:", sum(p.numel() for p in bahdanau.parameters()))
print("Luong(dot) params:", sum(p.numel() for p in luong_dot.parameters()))
print("Luong(general) params:", sum(p.numel() for p in luong_general.parameters()))

Bahdanau context shape: torch.Size([2, 16]) weights shape: torch.Size([2, 5])
Luong(dot) context shape: torch.Size([2, 16]) weights shape: torch.Size([2, 5])
Luong(general) context shape: torch.Size([2, 16]) weights shape: torch.Size([2, 5])
Bahdanau params: 528
Luong(dot) params: 0
Luong(general) params: 256
